In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

SOURCE_TABLE = f"{BRONZE}.bronze_enrollment"
TARGET_TABLE = f"{SILVER}.dim_patient"

In [0]:
latest_patient_df = (
    spark.table(SOURCE_TABLE)
         .select(
             "patient_id",
             "study_id",
             "site_id",
             "country_code",
             "randomization_date",
             "study_arm",
             "patient_status",
             "ingestion_timestamp"
         )
)

window_df = latest_patient_df.groupBy("patient_id") \
    .agg(
        F.max("ingestion_timestamp")
         .alias("latest_ingestion_timestamp")
    )

patient_df = (
    latest_patient_df.alias("src")
    .join(
        window_df.alias("w"),
        (
            (F.col("src.patient_id") == F.col("w.patient_id"))
            &
            (
                F.col("src.ingestion_timestamp")
                ==
                F.col("w.latest_ingestion_timestamp")
            )
        ),
        "inner"
    )
    .select(
        "src.patient_id",
        "src.study_id",
        "src.site_id",
        "src.country_code",
        "src.randomization_date",
        "src.study_arm",
        "src.patient_status"
    )
)

In [0]:
target = DeltaTable.forName(
    spark,
    TARGET_TABLE
)

(
    target.alias("t")
    .merge(
        patient_df.alias("s"),
        "t.patient_id = s.patient_id"
    )
    .whenMatchedUpdate(
        set={
            "study_id": "s.study_id",
            "site_id": "s.site_id",
            "country_code": "s.country_code",
            "randomization_date": "s.randomization_date",
            "study_arm": "s.study_arm",
            "patient_status": "s.patient_status"
        }
    )
    .whenNotMatchedInsert(
        values={
            "patient_id": "s.patient_id",
            "study_id": "s.study_id",
            "site_id": "s.site_id",
            "country_code": "s.country_code",
            "randomization_date": "s.randomization_date",
            "study_arm": "s.study_arm",
            "patient_status": "s.patient_status"
        }
    )
    .execute()
)

In [0]:
print(
    f"dim_patient records : "
    f"{spark.table(TARGET_TABLE).count()}"
)

display(
    spark.table(TARGET_TABLE)
)